In [ ]:
!pip install -q python-dotenv transformers datasets scikit-learn torch gensim sentencepiece

import os
import sys
import gc

from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/NLP'
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
import matplotlib.pyplot as plt
import seaborn as sns

from evaluation.create_confusion_matrix import ConfusionMatrixCreator
from data.data_processor import DataProcessor
from data.dataset import FoodDataset
from features.tfidf import TFIDF
from models.svm_classifier import FinalSVMClassifier
from models.dual_trainer import DualModelTrainer



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 64.4 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
load_dotenv(os.path.join(PROJECT_PATH, '.env'))
hf_token = os.getenv('HF_TOKEN')
processor = DataProcessor()
data = processor.process_data(
    train_path='csv/train.csv',
    val_path='csv/valid.csv',
    test_path='csv/test.csv',
    tokenize=False
)

train_df = data['train_df']
val_df   = data['val_df']
test_df  = data['test_df']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Χρήση συσκευής: {device}")

In [ ]:
hazard_weights = compute_class_weight('balanced', classes=np.unique(train_df['hazard_label']), y=train_df['hazard_label'].values)
product_weights = compute_class_weight('balanced', classes=np.unique(train_df['product_label']), y=train_df['product_label'].values)

tensor_hazard_weights = torch.tensor(hazard_weights, dtype=torch.float32).to(device)
tensor_product_weights = torch.tensor(product_weights, dtype=torch.float32).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, predictions, average='macro')}

LOCAL_TEMP_DIR = "/content/temp_models"
os.makedirs(LOCAL_TEMP_DIR, exist_ok=True)

In [ ]:
#BERT
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased")

train_dataset_bert = FoodDataset(train_df['clean_text'], train_df['hazard_label'], train_df['product_label'], tokenizer_bert)
val_dataset_bert = FoodDataset(val_df['clean_text'], val_df['hazard_label'], val_df['product_label'], tokenizer_bert)
dummy_labels = pd.Series([0] * len(test_df))
test_dataset_bert = FoodDataset(test_df['clean_text'], dummy_labels, dummy_labels, tokenizer_bert)

#HAZARD
model_bert_haz = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=10).to(device)
args_haz = TrainingArguments(output_dir=f"{LOCAL_TEMP_DIR}/res_bert_haz", eval_strategy="epoch", save_strategy="epoch",
                             learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
                             num_train_epochs=15, load_best_model_at_end=True, metric_for_best_model="macro_f1",
                             remove_unused_columns=False, label_names=['hazard_label'], save_total_limit=1)
trainer_bert_haz = DualModelTrainer(target_column='hazard_label', class_weights=tensor_hazard_weights, model=model_bert_haz, args=args_haz, train_dataset=train_dataset_bert, eval_dataset=val_dataset_bert, compute_metrics=compute_metrics)
trainer_bert_haz.train()
bert_hazard_probs = F.softmax(torch.tensor(trainer_bert_haz.predict(test_dataset_bert).predictions), dim=-1).numpy()
bert_haz_val_probs = F.softmax(torch.tensor(trainer_bert_haz.predict(val_dataset_bert).predictions), dim=-1).numpy()

#PRODUCT
model_bert_prod = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=22).to(device)
args_prod = TrainingArguments(output_dir=f"{LOCAL_TEMP_DIR}/res_bert_prod", eval_strategy="epoch", save_strategy="epoch",
                              learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
                              num_train_epochs=15, load_best_model_at_end=True, metric_for_best_model="macro_f1",
                              remove_unused_columns=False, label_names=['product_label'], save_total_limit=1)
trainer_bert_prod = DualModelTrainer(target_column='product_label', class_weights=tensor_product_weights, model=model_bert_prod, args=args_prod, train_dataset=train_dataset_bert, eval_dataset=val_dataset_bert, compute_metrics=compute_metrics)
trainer_bert_prod.train()
bert_product_probs = F.softmax(torch.tensor(trainer_bert_prod.predict(test_dataset_bert).predictions), dim=-1).numpy()
bert_prod_val_probs = F.softmax(torch.tensor(trainer_bert_prod.predict(val_dataset_bert).predictions), dim=-1).numpy()

del model_bert_haz, model_bert_prod, trainer_bert_haz, trainer_bert_prod, train_dataset_bert, val_dataset_bert
torch.cuda.empty_cache()
gc.collect()

In [ ]:
#RoBERTa
tokenizer_roberta = AutoTokenizer.from_pretrained("roberta-base")

train_dataset_rob = FoodDataset(train_df['clean_text'], train_df['hazard_label'], train_df['product_label'], tokenizer_roberta)
val_dataset_rob = FoodDataset(val_df['clean_text'], val_df['hazard_label'], val_df['product_label'], tokenizer_roberta)
test_dataset_rob = FoodDataset(test_df['clean_text'], dummy_labels, dummy_labels, tokenizer_roberta)

#HAZARD
model_rob_haz = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=10).to(device)
args_haz_rob = TrainingArguments(output_dir=f"{LOCAL_TEMP_DIR}/res_rob_haz", eval_strategy="epoch", save_strategy="epoch",
                                 learning_rate=1.5e-5, warmup_ratio=0.1, lr_scheduler_type="cosine", per_device_train_batch_size=16, per_device_eval_batch_size=16,
                                 num_train_epochs=15, load_best_model_at_end=True, metric_for_best_model="macro_f1",
                                 remove_unused_columns=False, label_names=['hazard_label'], save_total_limit=1)
trainer_rob_haz = DualModelTrainer(target_column='hazard_label', class_weights=tensor_hazard_weights, model=model_rob_haz, args=args_haz_rob, train_dataset=train_dataset_rob, eval_dataset=val_dataset_rob, compute_metrics=compute_metrics)
trainer_rob_haz.train()
roberta_hazard_probs = F.softmax(torch.tensor(trainer_rob_haz.predict(test_dataset_rob).predictions), dim=-1).numpy()
rob_haz_val_probs = F.softmax(torch.tensor(trainer_rob_haz.predict(val_dataset_rob).predictions), dim=-1).numpy()

#PRODUCT
model_rob_prod = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=22).to(device)
args_prod_rob = TrainingArguments(output_dir=f"{LOCAL_TEMP_DIR}/res_rob_prod", eval_strategy="epoch", save_strategy="epoch",
                                  learning_rate=1.5e-5, warmup_ratio=0.1, lr_scheduler_type="cosine", per_device_train_batch_size=16, per_device_eval_batch_size=16,
                                  num_train_epochs=15, load_best_model_at_end=True, metric_for_best_model="macro_f1",
                                  remove_unused_columns=False, label_names=['product_label'], save_total_limit=1)
trainer_rob_prod = DualModelTrainer(target_column='product_label', class_weights=tensor_product_weights, model=model_rob_prod, args=args_prod_rob, train_dataset=train_dataset_rob, eval_dataset=val_dataset_rob, compute_metrics=compute_metrics)
trainer_rob_prod.train()
roberta_product_probs = F.softmax(torch.tensor(trainer_rob_prod.predict(test_dataset_rob).predictions), dim=-1).numpy()
rob_prod_val_probs = F.softmax(torch.tensor(trainer_rob_prod.predict(val_dataset_rob).predictions), dim=-1).numpy()

del model_rob_haz, model_rob_prod, trainer_rob_haz, trainer_rob_prod, train_dataset_rob, val_dataset_rob
torch.cuda.empty_cache()
gc.collect()

Χρήση συσκευής: cuda


In [ ]:
# SVM(C=5.0)

tfidf = TFIDF(max_features=15000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df['clean_text'])
X_test_tfidf = tfidf.transform(test_df['clean_text'])

#HAZARD
svm_haz = FinalSVMClassifier(C=5.0)
svm_haz.train(X_train_tfidf, train_df['hazard_label'])
svm_hazard_probs = svm_haz.predict_proba(X_test_tfidf)

#PRODUCT
svm_prod = FinalSVMClassifier(C=5.0)
svm_prod.train(X_train_tfidf, train_df['product_label'])
svm_product_probs = svm_prod.predict_proba(X_test_tfidf)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

508

In [ ]:
W_BERT = 0.35
W_ROB  = 0.35
W_SVM  = 0.30

ens_haz_probs = (W_BERT * bert_hazard_probs) + (W_ROB * roberta_hazard_probs) + (W_SVM * svm_hazard_probs)
ens_prod_probs = (W_BERT * bert_product_probs) + (W_ROB * roberta_product_probs) + (W_SVM * svm_product_probs)

final_haz_preds = np.argmax(ens_haz_probs, axis=-1)
final_prod_preds = np.argmax(ens_prod_probs, axis=-1)

final_haz_text = data['le_hazard'].inverse_transform(final_haz_preds)
final_prod_text = data['le_product'].inverse_transform(final_prod_preds)

multimodal_ensemble = pd.DataFrame({
    'id': test_df.index,
    'hazard-category': final_haz_text,
    'product-category': final_prod_text
})

sub_path = os.path.join('csv', 'multimodal_ensemble_submission.csv')
multimodal_ensemble.to_csv(sub_path, index=False)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
X_val_tfidf = tfidf.transform(val_df['clean_text'])
svm_haz_val_probs = svm_haz.predict_proba(X_val_tfidf)
svm_prod_val_probs = svm_prod.predict_proba(X_val_tfidf)

W_BERT = 0.35
W_ROB  = 0.35
W_SVM  = 0.30

ens_haz_val_probs = (W_BERT * bert_haz_val_probs) + (W_ROB * rob_haz_val_probs) + (W_SVM * svm_haz_val_probs)
ens_prod_val_probs = (W_BERT * bert_prod_val_probs) + (W_ROB * rob_prod_val_probs) + (W_SVM * svm_prod_val_probs)

ens_haz_val_preds = np.argmax(ens_haz_val_probs, axis=-1)
ens_prod_val_preds = np.argmax(ens_prod_val_probs, axis=-1)

In [ ]:
W_BERT = 0.35
W_ROB  = 0.35
W_SVM  = 0.30

ens_haz_probs = (W_BERT * bert_hazard_probs) + (W_ROB * roberta_hazard_probs) + (W_SVM * svm_hazard_probs)
ens_prod_probs = (W_BERT * bert_product_probs) + (W_ROB * roberta_product_probs) + (W_SVM * svm_product_probs)

final_haz_preds = np.argmax(ens_haz_probs, axis=-1)
final_prod_preds = np.argmax(ens_prod_probs, axis=-1)

final_haz_text = data['le_hazard'].inverse_transform(final_haz_preds)
final_prod_text = data['le_product'].inverse_transform(final_prod_preds)

multimodal_ensemble = pd.DataFrame({
    'id': test_df.index,
    'hazard-category': final_haz_text,
    'product-category': final_prod_text
})

sub_path = os.path.join('csv', 'multimodal_ensemble_submission.csv')
multimodal_ensemble.to_csv(sub_path, index=False)

In [ ]:
# Hazard
cm_creator_haz = ConfusionMatrixCreator(
    train_df=train_df, 
    val_df=val_df, 
    target_col='hazard_label', 
    class_names=data['le_hazard'].classes_
)

cm_creator_haz.plot_predictions(
    y_true=val_df['hazard_label'].values,
    y_pred=ens_haz_val_preds,
    title="Confusion Matrix: Weighted Ensemble (Hazard Category)",
    filename="cm_ensemble_hazard.png"
)

#Product
cm_creator_prod = ConfusionMatrixCreator(
    train_df=train_df, 
    val_df=val_df, 
    target_col='product_label', 
    class_names=data['le_product'].classes_
)

cm_creator_prod.plot_predictions(
    y_true=val_df['product_label'].values,
    y_pred=ens_prod_val_preds,
    title="Confusion Matrix: Weighted Ensemble (Product Category)",
    filename="cm_ensemble_product.png",
    figsize=(16, 14),
    annot_size=8
)


models = ['Baseline SVM', 'BERT', 'RoBERTa', 'Weighted Ensemble']
scores = [0.7078, 0.7651, 0.7721, 0.7764]

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=models, y=scores, palette="viridis")

plt.ylim(0.65, 0.85)
plt.title("Ablation Study: Official ST1 Score Evolution", fontsize=16, fontweight='bold', pad=15)
plt.ylabel("Official ST1 Score (Test Set)", fontsize=14, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

for p in ax.patches:
    ax.annotate(f"{p.get_height():.4f}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                fontsize=13, fontweight='bold', color='black',
                xytext=(0, 10), textcoords='offset points')

filepath = os.path.join("images", "ablation_study_st1.png")
plt.savefig(filepath, bbox_inches='tight', dpi=300)
plt.show()